# MultiBBQ Image Synthesis: GPT-Image-1

This notebook synthesizes the **MultiBBQ** benchmark images using OpenAI's **GPT-Image-1** model (`model="gpt-image-1"`).

**Reads:** the prompt/template table produced by [`../gen_template.ipynb`](../gen_template.ipynb), expected at `../data/construction/multibbq_template_table.csv`. This table supplies the per-row generation prompts (`visual_only_ambig_prompt_w_position`, `visual_only_ambig_prompt_wo_position`, `visual_textual_prompt`) together with the contexts, questions, options and their masked variants.

**Writes:**
- Generated PNG images under `data/images/gpt_image_gen/visual_only/` and `data/images/gpt_image_gen/visual_language/`.
- A sidecar `.txt` file per image recording the prompt, contexts and Q&A used.
- Provenance CSV/JSON tables under `../data/metadata/gpt_image_gen/` (`multibbq_visual_only.*`, `multibbq_visual_language.*`) with an added `image_path` column.

> After running, move the produced `images/` folder to the project root as described in `README.md`.

**API credential:** requires an OpenAI API key. The key is read from the `OPENAI_API_KEY` environment variable (`os.environ["OPENAI_API_KEY"]`); set it in your shell before running rather than hardcoding it.

## Imports

Standard library plus `requests` and the OpenAI SDK used to call GPT-Image-1.

In [ ]:
import os
import json
import configparser
import base64

import requests
from openai import OpenAI



## Configure the GPT-Image-1 client and image helpers

Instantiate the OpenAI client (API key from the `OPENAI_API_KEY` environment variable) and define helpers:
- `gpt_image_gen(prompt)`: call the `gpt-image-1` API for a single 1024x1024, high-quality image.
- `download_and_save_image(...)` / `gen_and_save_images(...)`: decode the returned base64 image and write it to disk.

In [ ]:
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def gpt_image_gen(prompt: str):

    return client.images.generate(
        model="gpt-image-1",
        prompt=prompt,
        size="1024x1024",
        quality="high",
        moderation="low",
        n=1,
    )

def download_and_save_image(image_bytes, image_path):

    print(f'saving to {image_path}')
    with open(image_path, "wb") as image_file:
        image_file.write(image_bytes)


def gen_and_save_images(prompt: str, image_path: str):

    res = gpt_image_gen(prompt)
    b64_str = res.data[0].b64_json
    image_bytes = base64.b64decode(b64_str)
    os.makedirs(os.path.dirname(image_path), exist_ok=True)
    with open(image_path, "wb") as f:
        f.write(image_bytes)
    return image_path

## Load the prompt table

Load the template/prompt table from `../gen_template.ipynb`. The `options` columns are stored as stringified Python lists, so they are parsed back with `eval`. (These lines are commented out here; uncomment them to load `template_df` before running the generation cells below.)

In [ ]:
import pandas as pd

# template_df = pd.read_csv('../data/construction/multibbq_template_table.csv')
# template_df['options'] = template_df['options'].apply(eval)
# template_df['options_masked'] = template_df['options_masked'].apply(eval)


## Generate the *visual-only* image set

For each template row, build the visual-only prompt (with or without positional cues), call GPT-Image-1, and save the PNG to `data/images/gpt_image_gen/visual_only/` along with a `.txt` sidecar capturing the prompt, contexts and Q&A. Existing images are skipped. The resulting table (with an `image_path` column) is written to `../data/metadata/gpt_image_gen/multibbq_visual_only.{csv,json}`.

> ⚠️ **Writes to the released dataset.** The next cells overwrite `../data/metadata/gpt_image_gen/`.

In [ ]:
def generate_visual(df: pd.DataFrame, ambiguity: str = 'ambig', gen_img: bool = True, position: bool = True):
    visual_df = df.copy()
    image_paths = []
    
    for idx, row in visual_df.iterrows():
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, f'../data/images/gpt_image_gen/visual_only')
        if not os.path.isdir(image_dir):
            os.makedirs(image_dir)
        
        category = row['category']
        options = row['options']

        options_masked = row['options_masked']
        
        visual_only_a = 'unknown'
        
        if ambiguity == 'ambig':
            context = 'Context: ' + row['ambig_context']
            context_masked = 'Context(masked): ' + row['ambig_context_masked']
        else:
            context = 'Context: ' + row['ambig_context'] + ' ' + row['disambig_context']
            context_masked = 'Context(masked): ' + row['ambig_context_masked'] + '\n' +  row['disambig_context_masked']
        name1 = row['name1']
        name2 = row['name2']
        
        name1_modified = name1.replace(' ', '')
        name2_modified = name2.replace(' ', '')
        image_name = f'visual_only_{category}_q{row.q_id}_c{row.c_id}_{name1_modified}_{name2_modified}'

        qna = 'Negtive Question: ' + row['neg_q'] + ' ' + 'Ans: ' + visual_only_a + '\n' + 'Nonnegtive Question: ' + row['nonneg_q'] + ' ' + 'Ans: ' + visual_only_a + '\n' + 'Options: '+ ' -'.join(row['options'])
        qna_masked = 'Negtive Question(masked): ' + row['neg_q'] + ' ' + 'Ans(masked): ' + visual_only_a + '\n' + 'Nonnegtive Question(masked): ' + row['nonneg_q'] + ' ' + 'Ans(masked): ' + visual_only_a + '\n' + 'Options(masked): ' + ' -'.join(row['options_masked'])

        if position:
            prompt = row['visual_only_ambig_prompt_w_position']
        else:
            prompt = row['visual_only_ambig_prompt_wo_position']

        image_path = os.path.join(image_dir, f'{image_name}.png')
        image_paths.append(image_path)
        retry = 1
        for i in range(retry):
            if i != 0:
                image_path = image_path.replace('.png', f'_{i}.png')
            if not os.path.exists(image_path):
                with open(os.path.join(image_dir, f'{image_name}.txt'), 'w') as f:
                    f.write('Prompt: '+prompt+'\n\n')
                    try:
                        if gen_img:
                            gen_and_save_images(prompt, image_path=image_path)

                    except Exception as e:
                        print(e)
                        f.write(str(e))
                    f.write(context+'\n')
                    f.write(context_masked+'\n\n')
                    f.write(qna+'\n\n')
                    f.write(qna_masked+'\n')
    visual_df['image_path'] = image_paths        
    return visual_df



visual_data = generate_visual(template_df, 'ambig', True)
visual_data.to_csv('../data/metadata/gpt_image_gen/multibbq_visual_only.csv', index=False)
visual_data.to_json('../data/metadata/gpt_image_gen/multibbq_visual_only.json', orient='records')


## Generate the *visual-language (textual)* image set

Same flow for the visual+textual condition: build the `visual_textual_prompt` per row, generate and save each image to `data/images/gpt_image_gen/visual_language/` with its `.txt` sidecar, then write the table to `../data/metadata/gpt_image_gen/multibbq_visual_language.{csv,json}`.

In [ ]:
def generate_textual(df: pd.DataFrame, gen_img: bool = True, position: bool = True):
    textual_df = df.copy()
    image_paths = []

    for idx, row in textual_df.iterrows():
        image_dir = os.path.join(os.curdir, f'../data/images/gpt_image_gen/visual_language')
        if not os.path.isdir(image_dir):
            os.makedirs(image_dir)
        
        category = row['category']
        options = row['options']
        options_masked = row['options_masked']
        
        neg_a = options[row['neg_label_idx']]
        nonneg_a = options[row['nonneg_label_idx']]
        neg_a_masked = options_masked[row['neg_label_idx']]
        nonneg_a_masked = options_masked[row['nonneg_label_idx']]

        name1 = row['name1']
        name2 = row['name2']
        name1_modified = name1.replace(' ', '')
        name2_modified = name2.replace(' ', '')
        image_name = f'visual_language_{category}_q{row.q_id}_c{row.c_id}_{name1_modified}_{name2_modified}'

        context = 'Ambiguous Context: ' + row['ambig_context'] + '\n' +  'Disambiguating Context: '+ row['disambig_context']
        context_masked = 'Ambiguous Context(masked): ' + row['ambig_context_masked'] + '\n' +  'Disambiguating Context(masked): ' + row['disambig_context_masked']
        qna = 'Negtive Question: ' + row['neg_q'] + ' ' + 'Ans: ' + neg_a + '\n' + 'Nonnegtive Question: ' + row['nonneg_q'] + ' ' + 'Ans:'+ nonneg_a + '\n' + 'Options: '+' -'.join(row['options'])
        qna_masked = 'Negtive Question(masked): ' + row['neg_q'] + ' ' + 'Ans(masked): ' + neg_a_masked + '\n' + 'Nonnegtive Question(masked): '+ row['nonneg_q'] + ' ' + 'Ans(masked):' + nonneg_a_masked + '\n' + 'Options(masked): '+' -'.join(row['options_masked'])

        prompt = row['visual_textual_prompt']
    
        image_path = os.path.join(image_dir, f'{image_name}.png')
        image_paths.append(image_path)
        retry = 1
        for i in range(retry):
            if i != 0:
                image_path = image_path.replace('.png', f'_{i}.png')
        if not os.path.exists(image_path):
            with open(os.path.join(image_dir, f'{image_name}.txt'), 'w') as f:
                f.write('Prompt: '+prompt+'\n\n')
                try:
                    if gen_img:
                        gen_and_save_images(prompt, image_path=image_path)
                except Exception as e:
                    f.write(str(e))
                
                f.write(context+'\n')
                f.write(context_masked+'\n\n')
                f.write(qna+'\n\n')
                f.write(qna_masked+'\n')
    textual_df['image_path'] = image_paths
    return textual_df

textual_data = generate_textual(template_df, True)
textual_data.to_csv('../data/metadata/gpt_image_gen/multibbq_visual_language.csv', index=False)
textual_data.to_json('../data/metadata/gpt_image_gen/multibbq_visual_language.json', orient='records')
textual_data


## Done

Confirmation message pointing to the output image directories.

In [ ]:
print('Visual-only and visual-language image generation completed.' + '\n\n' + 'You can find the generated images in the ../data/images/gpt_image_gen/visual_only and ../data/images/gpt_image_gen/visual_language directories (the repository data/images/ tree).' + '\n\n')